## CIFAR 10 

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# -------------------------------------------------------------------
# Load the CIFAR-10 dataset (60,000 32x32 color images, 10 classes)
# -------------------------------------------------------------------
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normalize pixel values to [0,1] for better training stability
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Convert labels to one-hot encoding (e.g., class 3 -> [0,0,0,1,0,0,0,0,0,0])
y_train = keras.utils.to_categorical(y_train, num_classes=10)
y_test = keras.utils.to_categorical(y_test, num_classes=10)

# Create tf.data pipelines for batching and prefetching
batch_size = 64
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(50000).batch(batch_size).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_ds = test_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# -------------------------------------------------------------------
# Build a simple CNN with Conv2D, MaxPooling, Dropout, and Dense layers
# -------------------------------------------------------------------
model = keras.Sequential([
    # First convolutional block
    layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
    layers.BatchNormalization(),               # speeds up convergence
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),               # reduces spatial size to 16x16
    layers.Dropout(0.2),

    # Second convolutional block
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),               # reduces to 8x8
    layers.Dropout(0.3),

    # Third convolutional block
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),               # reduces to 4x4
    layers.Dropout(0.4),

    # Fully connected head
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')    # 10 classes
])

# Compile with Adam optimizer and categorical crossentropy loss
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# -------------------------------------------------------------------
# Train the model
# -------------------------------------------------------------------
epochs = 15
history = model.fit(train_ds,
                    validation_data=test_ds,
                    epochs=epochs)

# Evaluate on test set
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")


# Optional: plot training history
plt.plot(history.history['accuracy'], label='Train acc')
plt.plot(history.history['val_accuracy'], label='Val acc')
plt.legend()
plt.title('Classification Accuracy')
plt.show()

## Object Detection – Oxford‑IIIT Pets (bounding boxes)

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt

# -------------------------------------------------------------------
# Load the Food-101 dataset from TensorFlow Datasets
#
# Food-101 contains 101 food categories, with 101'000 images total.
# For each class: 750 training images (often noisy, real-world data)
# and 250 clean test images. All images are scaled to max 512px.
#
# We split the data manually: 80% of training images for training,
# 20% for validation.
# -------------------------------------------------------------------
# Download and prepare the dataset (takes a while the first time)
train_ds_raw, info = tfds.load(
    'food101',
    split='train',
    with_info=True,
    as_supervised=True          # returns (image, label) tuples
)

# Determine training / validation split sizes
total_train = info.splits['train'].num_examples      # 75,750 images
val_size = int(0.2 * total_train)                    # 15,150 images
train_size = total_train - val_size                  # 60,600 images

# Split the training set into train and validation
train_ds = train_ds_raw.take(train_size)             # first 60,600
val_ds   = train_ds_raw.skip(train_size)             # remaining 15,150

# -------------------------------------------------------------------
# Preprocessing functions
# -------------------------------------------------------------------
def preprocess(image, label):
    """
    Resize image to a fixed size (224x224) and normalize pixels to [0,1].
    Convert label to one-hot encoding for 101 classes.
    """
    # Resize the image to 224x224 (standard for many pre-trained models)
    image = tf.image.resize(image, (224, 224))
    # Normalise pixel values to [0,1] for stable training
    image = tf.cast(image, tf.float32) / 255.0
    # One-hot encode the label (101 classes)
    label = tf.one_hot(label, depth=101)
    return image, label

def augment(image, label):
    """
    Apply aggressive data augmentation to improve generalisation.
    Random horizontal flip, brightness, contrast, and saturation.
    """
    # Random horizontal flip (mirror the image)
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)

    # Random brightness adjustment (±20%)
    image = tf.image.random_brightness(image, max_delta=0.2)
    # Random contrast adjustment (0.8 – 1.2)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    # Random saturation adjustment (0.8 – 1.2)
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)

    # Clip values back to [0,1] after augmentation
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

# -------------------------------------------------------------------
# Create tf.data pipelines
# -------------------------------------------------------------------
batch_size = 64

# Training pipeline: preprocess, augment, shuffle, batch, prefetch
train_pipeline = (
    train_ds
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .map(augment,    num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(buffer_size=1024)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

# Validation pipeline: only preprocess, no augmentation
val_pipeline = (
    val_ds
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

# -------------------------------------------------------------------
# Build a CNN suitable for 101 classes with higher resolution images
#
# This model is deeper than the CIFAR-10 one, uses BatchNormalization
# after every convolution, and has a larger feature map count to
# capture fine-grained details.
# -------------------------------------------------------------------
model = keras.Sequential([
    # First convolutional block – high resolution, fewer filters
    layers.Conv2D(64, (3,3), activation='relu', padding='same', input_shape=(224,224,3)),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),                # 224 -> 112
    layers.Dropout(0.2),

    # Second block – increase filter depth
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),                # 112 -> 56
    layers.Dropout(0.3),

    # Third block – deeper filters
    layers.Conv2D(256, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),                # 56 -> 28
    layers.Dropout(0.4),

    # Fourth block – more capacity for fine-grained classification
    layers.Conv2D(512, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(512, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),                # 28 -> 14
    layers.Dropout(0.4),

    # Fifth block – highest level features
    layers.Conv2D(512, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(512, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),                # 14 -> 7
    layers.Dropout(0.5),

    # Fully connected head
    layers.Flatten(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(101, activation='softmax')    # 101 classes
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),  # Lower LR for larger model
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# -------------------------------------------------------------------
# Train the model
# -------------------------------------------------------------------
epochs = 30
history = model.fit(
    train_pipeline,
    validation_data=val_pipeline,
    epochs=epochs,
    verbose=1
)

# Evaluate on the validation set
val_loss, val_acc = model.evaluate(val_pipeline)
print(f"Validation accuracy: {val_acc:.4f}")

# Plot training history
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train acc')
plt.plot(history.history['val_accuracy'], label='Val acc')
plt.title('Accuracy')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train loss')
plt.plot(history.history['val_loss'], label='Val loss')
plt.title('Loss')
plt.legend()
plt.tight_layout()
plt.show()

/Users/rafael/projects/ml_bootcamp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dl Completed...: 0 url [00:00, ? url/s]
Dl Completed...:   0%|          | 0/1 [05:11<?, ? url/s]
Extraction completed...: 0 file [05:11, ? file/s]
Dl Completed...:   0%|          | 0/1 [05:11<?, ? url/s]


KeyboardInterrupt: 

## Segmentation

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds

# -------------------------------------------------------------------
# Load Oxford-IIIT Pets dataset with segmentation masks.
# Mask values: 1 = pet, 2 = background, 3 = border (we'll treat as binary pet/background)
# -------------------------------------------------------------------
dataset, info = tfds.load('oxford_iiit_pet', split='train+test', with_info=True)
train_size = int(0.8 * info.splits['train+test'].num_examples)
val_size = info.splits['train+test'].num_examples - train_size
train_ds, val_ds = tfds.load('oxford_iiit_pet', split=[f'train[:{train_size}]', f'train[{train_size}:]'])

def preprocess(sample):
    """Resize image and mask to 128x128, normalize image, binarize mask."""
    image = sample['image']
    mask = sample['segmentation_mask']

    # Resize image and mask (nearest neighbour for mask to preserve labels)
    image = tf.image.resize(image, (128, 128))
    mask = tf.image.resize(mask, (128, 128), method='nearest')

    # Normalize image to [0,1]
    image = tf.cast(image, tf.float32) / 255.0

    # Convert mask: 1 (pet) -> 1, everything else -> 0 (background)
    mask = tf.cast(tf.equal(mask, 1), tf.float32)   # binary mask

    return image, mask

def augment(image, mask):
    """Random horizontal flip on both image and mask."""
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)
    return image, mask

batch_size = 16
train_ds = train_ds.map(preprocess).map(augment).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).batch(batch_size).prefetch(tf.data.AUTOTUNE)

# -------------------------------------------------------------------
# Build a U‑Net style CNN for binary segmentation.
# Encoder (downsampling) + Decoder (upsampling) with skip connections.
# -------------------------------------------------------------------
def unet(input_size=(128,128,3)):
    inputs = keras.Input(shape=input_size)

    # Encoder (contracting path)
    # Block 1
    c1 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2,2))(c1)

    # Block 2
    c2 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2,2))(c2)

    # Block 3
    c3 = layers.Conv2D(256, (3,3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(256, (3,3), activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2,2))(c3)

    # Block 4 (bottleneck)
    c4 = layers.Conv2D(512, (3,3), activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(512, (3,3), activation='relu', padding='same')(c4)

    # Decoder (expanding path) with skip connections
    # Up block 1
    u5 = layers.UpSampling2D((2,2))(c4)
    u5 = layers.Conv2D(256, (2,2), activation='relu', padding='same')(u5)
    u5 = layers.concatenate([u5, c3])          # skip connection
    c5 = layers.Conv2D(256, (3,3), activation='relu', padding='same')(u5)
    c5 = layers.Conv2D(256, (3,3), activation='relu', padding='same')(c5)

    # Up block 2
    u6 = layers.UpSampling2D((2,2))(c5)
    u6 = layers.Conv2D(128, (2,2), activation='relu', padding='same')(u6)
    u6 = layers.concatenate([u6, c2])
    c6 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(u6)
    c6 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(c6)

    # Up block 3
    u7 = layers.UpSampling2D((2,2))(c6)
    u7 = layers.Conv2D(64, (2,2), activation='relu', padding='same')(u7)
    u7 = layers.concatenate([u7, c1])
    c7 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(u7)
    c7 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(c7)

    # Output layer: 1 channel with sigmoid for binary segmentation
    outputs = layers.Conv2D(1, (1,1), activation='sigmoid')(c7)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

model = unet()
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy', keras.metrics.IoU(num_classes=2, target_class_ids=[1])])

model.summary()

# -------------------------------------------------------------------
# Train the segmentation model
# -------------------------------------------------------------------
epochs = 15
history = model.fit(train_ds,
                    validation_data=val_ds,
                    epochs=epochs)

# Display one prediction (after training)
for image, mask in val_ds.take(1):
    pred_mask = model.predict(image)
    # pred_mask is probability map; threshold at 0.5
    pred_mask = (pred_mask > 0.5).astype('float32')
    print("Prediction shape:", pred_mask.shape)
    # (Visualisation code would go here)